# Day 24: Integrating Docling for Complex PDF Parsing

Welcome to Day 24 of your AI Engineering journey! In Phase 2, we are establishing robust Vector Databases & RAG Foundations.

## Core Theory (Just-in-Time)

**Why Docling?**
In a production RAG system, your documents are rarely clean text files. They are often complex PDFs, Word documents, or PowerPoints containing embedded tables, images, multiple columns, and headers/footers.
Standard text extraction libraries (like `PyPDF2` or `pdfplumber`) often fail dramatically on these layouts. They read text line-by-line, which destroys the semantic structure of a multi-column page or a nested table. If you chunk and embed a broken table, the LLM will hallucinate when queried about that data.

**How Docling Works:**
`Docling` (by IBM) uses advanced layout analysis (often powered by vision models underneath) to semantically understand the document structure. It accurately reconstructs tables, headings, and paragraphs, and exports them into structured Markdown. Markdown is the optimal format for LLMs because it explicitly preserves hierarchical structure (e.g., `# Heading 1`, `| Table Header |`) while remaining extremely token-efficient.

**The Pipeline:**
1. Ingest complex PDF (with tables).
2. Process with `Docling` to output structured Markdown.
3. Chunk the Markdown using a Markdown-aware text splitter.
4. Vectorize and store in a Vector DB (like Qdrant) for retrieval.

**AI Security Implications:**
- **PII Redaction:** Parsed documents often contain sensitive information (names, emails, SSNs). Always redact this information *before* embedding and storing it in a vector database to prevent data leaks.
- **Prompt Injection:** Complex PDFs can harbor adversarial text hidden in white font or tables designed to hijack the LLM prompt. Sanitizing output and structuring it as markdown helps mitigate some injection risks, but robust input validation is necessary.
- **Fallbacks:** External OCR or parsing APIs can fail or rate-limit. Always implement fallback mechanisms (like a try/except returning a safe empty state or basic text extraction) to ensure system reliability.


## Code Implementation

Below is a tiered progression of parsing complex documents with Docling, moving from a basic script to a production-ready, secure implementation.

### 1. Basic Implementation
Isolate the core concept with minimal boilerplate.

In [1]:
from docling.document_converter import DocumentConverter

def basic_parse(url: str) -> str:
    # Minimal boilerplate for extracting markdown
    converter = DocumentConverter()
    result = converter.convert(url)
    return result.document.export_to_markdown()

# Example usage
# markdown_output = basic_parse("https://arxiv.org/pdf/2408.09869.pdf")
# print(markdown_output[:100])

/app/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 2. Medium Implementation
Emphasize clean OOP, state management, and how objects interact, along with a simple fallback.

In [2]:
from docling.document_converter import DocumentConverter

class DocumentParser:
    def __init__(self):
        self.converter = DocumentConverter()
        
    def parse_to_markdown(self, url: str) -> str:
        try:
            result = self.converter.convert(url)
            return result.document.export_to_markdown()
        except Exception as e:
            # Fallback mechanism in case of failure
            print(f"Error parsing document: {e}")
            return "Error: Document parsing failed."

# Example usage
# parser = DocumentParser()
# parsed_text = parser.parse_to_markdown("https://arxiv.org/pdf/2408.09869.pdf")

### 3. Advanced Implementation
Production-grade implementation with strict type hinting, docstrings, exact imports, robust error handling, Pydantic v2 validation, and AI Security (PII redaction).

In [3]:
import logging
import re
from typing import Optional
from docling.document_converter import DocumentConverter
from pydantic import BaseModel, Field, ValidationError

# Setup production-grade logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class ParsedDocument(BaseModel):
    """Schema for a successfully parsed document."""
    source_url: str = Field(..., description="The original URL or path of the document.")
    markdown_content: str = Field(..., description="The extracted markdown content.")

class SecureDocumentParser:
    """Production-ready OOP parser with security redaction."""
    def __init__(self):
        self.converter = DocumentConverter()
        
    def redact_pii(self, text: str) -> str:
        """Simple regex-based PII redaction for emails and phone numbers."""
        text = re.sub(r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', '[REDACTED_EMAIL]', text)
        text = re.sub(r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b', '[REDACTED_PHONE]', text)
        return text

    def parse_pdf_to_markdown(self, source_url: str) -> Optional[ParsedDocument]:
        """
        Parses a PDF document into structured Markdown using Docling.
        Includes error handling, fallbacks, and PII redaction.
        """
        logger.info(f"Initializing Docling DocumentConverter...")
        try:
            logger.info(f"Converting document from: {source_url}")
            result = self.converter.convert(source_url)
            md_text = result.document.export_to_markdown()
            
            # Apply AI Security: Redact PII
            safe_md_text = self.redact_pii(md_text)
            
            # Validate with Pydantic
            parsed_doc = ParsedDocument(
                source_url=source_url,
                markdown_content=safe_md_text
            )
            logger.info("Document successfully parsed, secured, and validated.")
            return parsed_doc
            
        except ValidationError as ve:
            logger.error(f"Data validation error for {source_url}: {ve}")
            return None
        except Exception as e:
            logger.error(f"Failed to parse document {source_url}. Fallback triggered. Error: {str(e)}")
            return None

# Example Usage
# We use a sample URL supported by Docling's typical test cases
secure_parser = SecureDocumentParser()
SAMPLE_URL = "https://arxiv.org/pdf/2408.09869.pdf"
parsed_result = secure_parser.parse_pdf_to_markdown(SAMPLE_URL)

if parsed_result:
    print("\n--- Extracted & Secured Markdown Snippet ---\n")
    print(parsed_result.markdown_content[:500] + "\n... [Truncated]")


2026-08-22 08:11:36,697 - INFO - Initializing Docling DocumentConverter...


2026-08-22 08:11:36,698 - INFO - Converting document from: https://arxiv.org/pdf/2408.09869.pdf


2026-08-22 08:11:36,935 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]


2026-08-22 08:11:37,093 - INFO - Going to convert document batch...


2026-08-22 08:11:37,094 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 1f82f44452ba377fa63e310478792c2e


...

[WARNING] 2026-08-22 08:11:54,548 [RapidOCR] main.py:132: The text detection result is empty


2026-08-22 08:11:54,561 - WARNING - RapidOCR returned empty result!


2026-08-22 08:13:09,001 - INFO - Finished converting document 2408.09869.pdf in 92.30 sec.


2026-08-22 08:13:09,118 - INFO - Document successfully parsed, secured, and validated.



--- Extracted & Secured Markdown Snippet ---

<!-- image -->

## Docling Technical Report

## Version 1.0

Christoph Auer Maksym Lysak Ahmed Nassar Michele Dolfi Nikolaos Livathinos Panos Vagenas Cesar Berrospi Ramis Matteo Omenetti Fabian Lindlbauer Kasper Dinkla Lokesh Mishra Yusik Kim Shubham Gupta Rafael Teixeira de Lima Valery Weber Lucas Morin Ingmar Meijer Viktor Kuropiatnyk Peter W. J. Staar

AI4K Group, IBM Research R¨ uschlikon, Switzerland

## Abstract

This technical report introduces Docling , an easy to use, self-contained, MI
... [Truncated]


## Common Pitfalls

1. **OCR Overhead vs. Native Text:** If a PDF has native digital text, basic parsers are fast. Docling applies deeper layout analysis, which can be computationally expensive (and slower) but yields far higher accuracy for tables and columns. In high-throughput systems, pre-filter documents to decide if heavy layout parsing is necessary.
2. **Token Limits and Table Chunking:** While Markdown tables are great, a massive table with 1000 rows will exceed context windows when chunked poorly. Even with Markdown, you must use intelligent splitting strategies (like LangChain's `MarkdownHeaderTextSplitter`) to keep chunks semantically cohesive.
3. **Hallucinating APIs:** Many developers assume LLMs can parse raw PDFs directly. While multimodal models can read image slices of a PDF, structured conversion to text (via Docling) remains vastly cheaper, faster, and more deterministically searchable in a Vector DB.

## Practical Lab / Homework

**Your Task:** 
1. Take the Markdown output generated above.
2. Use LangChain's `MarkdownHeaderTextSplitter` to split the text semantically based on headers.
3. Store the resulting chunks into a local Qdrant memory instance.
4. Query the points to verify successful insertion.

Below is the complete, working production-grade reference implementation for your lab.

**Bonus:** Record a brief 2-3 minute async video walkthrough of your design decisions, explaining how you handled potential parsing fallbacks and security considerations.

In [4]:
import uuid
from typing import List
from langchain_text_splitters import MarkdownHeaderTextSplitter
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

def process_and_index_markdown(markdown_text: str, collection_name: str = "docling_parsed_docs") -> None:
    """
    Splits markdown text via headers and indexes it into a local Qdrant collection using a dummy vector.
    
    Args:
        markdown_text (str): The markdown text to process.
        collection_name (str): The name of the Qdrant collection.
    """
    logger.info("Splitting markdown by headers...")
    # Define headers to split on
    headers_to_split_on = [
        ("#", "Header 1"),
        ("##", "Header 2"),
        ("###", "Header 3"),
    ]
    
    markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
    splits = markdown_splitter.split_text(markdown_text)
    
    logger.info(f"Generated {len(splits)} chunks.")
    if not splits:
        logger.warning("No splits generated. Text might be too short or lack headers.")
        return

    logger.info("Initializing in-memory Qdrant Client...")
    client = QdrantClient(":memory:")
    
    # Create collection
    # We use vector size 384 assuming a standard small embedding model like all-MiniLM-L6-v2
    # However, to avoid an external embedding dependency for this strict lab setup, 
    # we will use zeroed dummy vectors to demonstrate the architectural indexing flow.
    vector_size = 3
    client.recreate_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE),
    )
    
    logger.info("Preparing points for Qdrant...")
    points = []
    for i, split in enumerate(splits):
        # For demonstration without a live embedding API, use a dummy vector
        dummy_vector = [0.1, 0.2, 0.3]
        
        point = PointStruct(
            id=str(uuid.uuid4()),
            vector=dummy_vector,
            payload={
                "page_content": split.page_content,
                "metadata": split.metadata
            }
        )
        points.append(point)
        
    # Upsert points
    client.upsert(
        collection_name=collection_name,
        points=points
    )
    logger.info(f"Successfully indexed {len(points)} chunks into Qdrant collection '{collection_name}'.")
    
    # Query back to verify using the modern query_points API
    query_results = client.query_points(
        collection_name=collection_name,
        query=[0.1, 0.2, 0.3], # Dummy query matching the vector
        limit=1
    )
    
    print("\n--- Verification Query Result ---")
    if query_results.points:
        top_point = query_results.points[0]
        print(f"ID: {top_point.id}")
        print(f"Metadata: {top_point.payload.get('metadata', {})}")
        content = top_point.payload.get('page_content', '')
        print(f"Content snippet: {content[:100]}...")
    else:
        print("No results found.")

# Execute the lab integration
fallback_markdown = "# Dummy Document\n\nThis is a fallback document in case the parsing failed.\n\n## Section 1\n\nContent here."

try:
    if 'parsed_result' in globals() and parsed_result and parsed_result.markdown_content:
        content_to_index = parsed_result.markdown_content
    else:
        content_to_index = fallback_markdown
except NameError:
    content_to_index = fallback_markdown

process_and_index_markdown(content_to_index)



2026-08-22 08:13:11,848 - INFO - Splitting markdown by headers...


2026-08-22 08:13:11,852 - INFO - Generated 22 chunks.


2026-08-22 08:13:11,853 - INFO - Initializing in-memory Qdrant Client...


/tmp/ipykernel_32469/115416132.py:39: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(
2026-08-22 08:13:11,874 - INFO - Preparing points for Qdrant...


2026-08-22 08:13:11,879 - INFO - Successfully indexed 22 chunks into Qdrant collection 'docling_parsed_docs'.



--- Verification Query Result ---
ID: 2e85b5f1-7ce3-4aca-9b50-1c4aa8575521
Metadata: {'Header 2': 'Baselines for Object Detection'}
Content snippet: In Table 2, we present baseline experiments (given in mAP) on Mask R-CNN [12], Faster R-CNN [11], an...


## Reference Links
- [Docling Official Documentation](https://ds4sd.github.io/docling/)
- [LangChain MarkdownHeaderTextSplitter](https://python.langchain.com/docs/modules/data_connection/document_transformers/markdown_header_metadata/)
- [Qdrant Python Client](https://qdrant.tech/documentation/concepts/search/)